# Marketing Campaign A/B Testing

## End-to-End Statistics & Analytics Project

This notebook is designed to satisfy the DMV Core Tech project requirements.

### Sections
1. Import Libraries
2. Load Dataset
3. Data Cleaning
4. Exploratory Data Analysis
5. Summary Statistics
6. Distribution Comparison
7. Visualizations
8. Hypothesis Testing
9. Confidence Intervals
10. Effect Size (Cohen's d)
11. Power Analysis
12. Business Report & Recommendations


In [ ]:
# Install (if needed)
# !pip install statsmodels scipy seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import ttest_ind
from statsmodels.stats.weightstats import DescrStatsW, CompareMeans
from statsmodels.stats.power import TTestIndPower

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns",None)


## Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("marketing_AB.csv")
df.head()

## Data Cleaning

In [ ]:
df.columns=["Index","User_ID","Campaign","Converted",
              "Total_Ads","Most_Ads_Day","Most_Ads_Hour"]

df["Converted"]=df["Converted"].astype(int)

print(df.info())
print(df.isnull().sum())
print("Duplicates:",df.duplicated().sum())
df.head()

## Exploratory Analysis

In [ ]:
display(df.describe())
display(df.describe(include='object'))

display(df.groupby("Campaign").agg({
    "Converted":["count","sum","mean"],
    "Total_Ads":["mean","median","std","min","max"]
}))

## Visualizations

In [ ]:

plt.figure(figsize=(6,4))
sns.countplot(data=df,x="Campaign")
plt.title("Campaign Distribution")
plt.show()

plt.figure(figsize=(6,4))
sns.barplot(data=df,x="Campaign",y="Converted",estimator=np.mean)
plt.title("Conversion Rate")
plt.show()

plt.figure(figsize=(7,4))
sns.boxplot(data=df,x="Campaign",y="Total_Ads")
plt.show()

plt.figure(figsize=(7,4))
sns.violinplot(data=df,x="Campaign",y="Total_Ads")
plt.show()

plt.figure(figsize=(7,4))
sns.histplot(data=df,x="Total_Ads",hue="Campaign",kde=True)
plt.show()

plt.figure(figsize=(7,4))
sns.countplot(data=df,x="Most_Ads_Day",hue="Campaign")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(8,4))
sns.lineplot(data=df.groupby("Most_Ads_Hour")["Converted"].mean().reset_index(),
             x="Most_Ads_Hour",y="Converted")
plt.show()

plt.figure(figsize=(6,6))
df["Campaign"].value_counts().plot.pie(autopct="%1.1f%%")
plt.ylabel("")
plt.show()


## Hypothesis Testing

In [ ]:
campaign_a=df[df["Campaign"]=="ad"]["Converted"]
campaign_b=df[df["Campaign"]=="psa"]["Converted"]

print("H0 : Mean conversion rate is equal")
print("H1 : Mean conversion rate is different")

levene=stats.levene(campaign_a,campaign_b)
print("Levene Test:",levene)

t_stat,p_value=ttest_ind(campaign_a,campaign_b,equal_var=True)

print("T Statistic:",t_stat)
print("P-value:",p_value)

alpha=0.05

if p_value<alpha:
    print("Reject H0")
else:
    print("Fail to Reject H0")


## 95% Confidence Interval

In [ ]:
cm=CompareMeans(DescrStatsW(campaign_a),DescrStatsW(campaign_b))
print(cm.tconfint_diff(alpha=0.05,usevar='pooled'))

## Cohen's d

In [ ]:
mean1=np.mean(campaign_a)
mean2=np.mean(campaign_b)

sd1=np.std(campaign_a,ddof=1)
sd2=np.std(campaign_b,ddof=1)

n1=len(campaign_a)
n2=len(campaign_b)

pooled=np.sqrt(((n1-1)*sd1**2+(n2-1)*sd2**2)/(n1+n2-2))

d=(mean1-mean2)/pooled

print("Cohen's d:",d)

if abs(d)<0.2:
    print("Negligible Effect")
elif abs(d)<0.5:
    print("Small Effect")
elif abs(d)<0.8:
    print("Medium Effect")
else:
    print("Large Effect")


## Power Analysis

In [ ]:
analysis=TTestIndPower()

effect=abs(d)

power=analysis.solve_power(effect_size=effect,
                           nobs1=n1,
                           ratio=n2/n1,
                           alpha=0.05)

required=analysis.solve_power(effect_size=effect,
                              power=0.80,
                              alpha=0.05)

print("Post Hoc Power:",power)
print("Required Sample Size:",required)


# Final Report Template

## Statistical Findings
- Report p-value
- Report confidence interval
- Report Cohen's d
- Report power

## Business Interpretation
- Which campaign performed better?
- Is the improvement statistically significant?
- Is it practically significant?

## Recommendation
Roll out the better-performing campaign if:
- p-value < 0.05
- Effect size is meaningful
- Confidence interval supports improvement

## Limitations
- Binary conversion metric
- One experiment
- External factors not measured

## Next Steps
- Increase sample size
- Test more campaign variants
- Segment users
- Monitor long-term conversion
